# S1 2024 COMPSCI 714 - Tutorial 3: Neural Network Hyperparameters tuning and Regularisation

Welcome to Tutorial 3! This tutorial covers basics of hyperparameters tuning and regularisation for Neural Networks. 

*Disclaimer: some parts of the code and text used in this Notebook is directly reused or adapted from Aurélien Géron's notebooks https://github.com/ageron/handson-ml3/blob/main/10_neural_nets_with_keras.ipynb, https://github.com/ageron/handson-ml3/blob/main/11_training_deep_neural_networks.ipynb and his book "Hands-on Machine Learning with Scikit-Learn, Keras and Tensorflow, Ed.3", more particulary from Chapters 10 and 11.*

**Important note following Tutorial 2**:  
We used plots to view the evolution of the loss and accuracy for a training run. You can also use TensorBoard to visualise the learning curves during training, compare curves and statistics between multiple runs, visualise the computation graph, analyse training statistics, view images generated by your model, visualise complex multidimensional data projected down in 3D, and automatically clustered, profile your network (i.e., meanse the speed to identify bottlenecks), and more!  
If you are using TensorFlow, TensorBoard is automatically installed with it. If you wish to use it in Colab, you will need to install the TensorBoard plug-in by runing the following command:
```
pip install -q -U tensorboard-plugin-profile
```
We will not cover TensorBoard in the tutorials, but you can learn to use it with [the TensorBoard "Getting Started" notebook](https://www.tensorflow.org/tensorboard/get_started).  You can also use TensorBoard with PyTorch. Your can learn more about it with [the "How to use TensorBoard with Pytorch" notebook](https://pytorch.org/tutorials/recipes/recipes/tensorboard_with_pytorch.html).

Let's get started with tutorial 3!  
First of all, we need to import some libraries we will use in this tutorial:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

In [ ]:
import tensorflow as tf

If the `keras_tuner` module is not already installed on your machine, you should install it before runing the following cell. If you followed the requirement file provided at the start of the semester, it should already be installed. 

In [ ]:
import keras_tuner as kt # Library used for tuning hyperparameters with keras

**Note**: We will focus on using Keras/TensorFlow to perform hyperparameters tuning and regularisation in this tutorial. However, we will provide links along the way to extra external notebooks you can use if you wish to learn how to perform similar tasks with PyTorch. 

## I. Tuning neural network hyperparameters with Keras/Tensorflow

### I.1. Introduction to hyperparameter tuning

When building a Neural Network, there are a lot of choices to be made (e.g., number of layers, number of units per layer, activation functions, optimiser, learning rate, etc.). All this flexibility enables to build various models adapted to a wide variaty of tasks, but it results in a lot of hyperparameters needing to be tuned (i.e., their value needs to be adjusted to optimise the model performance and training). Two different sets of choices for the hyperparameters values can lead to very different results in term of model performance and training efficiency.  

As discussed in the lectures, there exists some intuitions about the effect of each hyperparameter, but the *art* of training a neural network remains an highly empirical one. Tuning hyper-parameters usually comes down to trying out several combinations and evaluating which one leads to the best results.  
Finding a satisfactory combination is usually done using *search* algorithms. We first define the *search space* (i.e., the space of possible values each hyperparameter can take) and then use search algorithms such as *grid search* or *random search* to find a "good" combination within the defined space. 

**Note**: The optimisation (i.e., training) of a neural network's parameters (i.e., weights) is also a search problem. The goal is to search for a combination of weight values which minimises the objective function (i.e., the loss/error.cost function). 

### I.2. Hyperparameter tuning with Keras Tuner

We will use the Fashion MNIST dataset again in this tutorial. Let's load it and split it into train and test sets.

In [ ]:
fashion_mnist = tf.keras.datasets.fashion_mnist.load_data()
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist
X_train, y_train = X_train_full[:-5000], y_train_full[:-5000]
X_valid, y_valid = X_train_full[-5000:], y_train_full[-5000:]

We will use the library `keras_tuner` (imported at the top of the notebook), which is the hyperparameter tuning library for Keras.

**Note**: Scikit-Learn also provides tools to perform search and therefore hyperparameter tuning. You can learn more about them [here](https://scikit-learn.org/stable/modules/grid_search.html). If you are training a Keras model, you will first need to convert it to a Scikit-Learn estimator before being able to use these tools. To do so, you can use the `KerasRegressor` and `KerasClassfier` wrapper classes from the [SciKeras library](https://adriangb.com/scikeras/stable/).

#### I.2.i. Specifying the search space

To use `keras_tuner`, you first need to define a function which builds, compiles, and returns a Keras model. The function must take a `kt.HyperParameters` object as an argument. This parameter will be passed by the tuner to the function to specify the values of hyperparameters used to build a model at a specific tuning iteration. This parameter is also used to specify the type (integer, float, string, etc.) and the range of possible values for each hyperparameters expected to build the model. 

**Todo**: Such a function is defined in the following cell. Have a look at it and discuss with your classmates what you think each part of the function does, in relation to what was explained previously.  
Once you have identified the different parts, you can run the cell and the extra explanation below. 

In [ ]:
def build_model(hp):
    n_hidden = hp.Int("n_hidden", min_value=0, max_value=8, default=2)
    n_neurons = hp.Int("n_neurons", min_value=16, max_value=256)
    learning_rate = hp.Float("learning_rate", min_value=1e-4, max_value=1e-2,
                             sampling="log")
    optimizer = hp.Choice("optimizer", values=["sgd", "adam"])
    if optimizer == "sgd":
        optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate)
    else:
        optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Flatten())
    for _ in range(n_hidden):
        model.add(tf.keras.layers.Dense(n_neurons, activation="relu"))
    model.add(tf.keras.layers.Dense(10, activation="softmax"))
    model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer,
                  metrics=["accuracy"])
    return model

The first part of the function defines the following hyperparameters.
- The **number of hidden layers**, `n_hidden`: the function call `hp.Int("n_hidden", min_value=0, max_value=8, default=2)` checks whether an integer hyperparameter named, "`n_hidden`" is present in the `HyperParameters` object `hp`. If so, it returns its value. Else, it registers a new hyparameter names "`n_hidden`", whose possible values range from 0 to 8 (inclusive), and it returns its default value (here, 2). When no default value is set, the minimum value is returned.
- The **number of units** (i.e., neurons) per hidden layer `n_neurons`: it is defined a similar way as `n_hidden`.
- The **learning rate** for the optimiser, `learning_rate`: the learning rate is registered as a float value, ranging from $10^{-4}$ to $10^{-2}$ with a logarithmic sampling (i.e., the learning rate will be sampled equally accross different orders of magnitude).
- The **optimisation algorithm** used to train the network, `optimizer`: it can take two possible values: `sgd` (default value as this is the first value) or `adam`. Depending on the value of `optimizer`, the function will train the model using a `SGD` or a `Adam` optimiser with the specified learning rate.

The second part of the function defines the model using the hyperparameter values and the `Sequential` API. You can see that the model starts with a `Flatten` layer to flatten the inputs, followed by a number of hidden layers determined by the `n_hidden` hyperparameter, each hidden layer using the ReLu activation function, and finally an output layer with 10 neurons (one per class) and a softmax activation function.

The model is compiled and returned at the end of the function.

**Note**: When tuning hyperparameters, you cannnot test all possible hyperparameters for every possible values they can take. You need to make "manual" choices about:
- What hyperparameters you want to tune (e.g., we could have also tried different activation functions in the hidden layers). The learning rate is usually one important hyperparameter to tune in any situation. 
- The range of values tested for each hyperparameter. You can look in the literature what values are usually used and start from there. As you gain more practice and experience with tuning neural networks, you will also get some more expert knowledge about what range of values are likely to yield good results. You can also start by runing a *sensitivity analysis* to evaluate what hyperparameters have the most influence on the performance of the model, and focus on tuning only these ones.

You can have a look at the different `HyperParameters` methods available with Keras to define different types of hyperparameters and their range in the [`HyperParameters` class documentation](https://keras.io/api/keras_tuner/hyperparameters/#hyperparameters-class).



#### I.2.ii. Random tuner

Now that you understand the `build_model` function better, you can use it to to perform a search. Let's start by performing a basic random search. This search will test hyperparameter values randomly within the specified range. We can use the `kt.RandomSearch` tuner and pass the `build_model` function to the constructor (i.e., the method used to created an instance of the `kt.RandomSearch` class). Then, the `search` method can be called to perform the search.  

**Todo**: Before runing it, have a look at the following cell. How many sets of hyperparameter values (i.e., versions of the model) will be evaluated? How many epochs will be run to train each version of the model?  
The following cell will take 2 to 3 minutes to run on a classic laptop. Note that if you tune larger models with a lot of trials (i.e., a lot of different combinations of hyperparameter values ), it will take much longer.

In [ ]:
random_search_tuner = kt.RandomSearch(
    build_model, objective="val_accuracy", max_trials=5, overwrite=False,
    directory="my_fashion_mnist", project_name="my_rnd_search")
random_search_tuner.search(X_train, y_train, epochs=10,
                           validation_data=(X_valid, y_valid))

Let's see what happened in the previous cell in more details:
- The `RandomSearch` tuner first gathers all the hyperparameter specifications by calling `build_model()` with an empty `Hyperparameters` object (this is happening "under the hood").
- Then, it runs a number of trials, here 5. For each trial, it builds a model with hyperparameter values sampled randomly within their respective ranges. Note that the argument used to specify the number of trials to be run is called `max_trials`. This is because it represents the total number of trials to test *at most*. The search might stop early if the search space (i.e., the number of possible combinations of hyperparameter values) has been exhausted before reaching this maximum number of trials.
- In each trial, the tuner trains a model for 10 epochs and saves it to a subdirectory of the "*my_fashion_mnist/my_rnd_search*" directory. Since the `overwrite` argument is set to `True`, the *my_rnd_search* directory is overwritten (i.e., deleted and written again) before training starts. If you run the following cell where `overwrite` argument is set to `False`, and `max_trials=10`, the tuner will run 5 additional trial, starting from where it left off (i.e., it will not overwrite the 5 trials it run before). This is useful if you do not want to run all the trials in one go. 

In [ ]:
random_search_tuner = kt.RandomSearch(
    build_model, objective="val_accuracy", max_trials=10, overwrite=False,
    directory="my_fashion_mnist", project_name="my_rnd_search", seed=42)
random_search_tuner.search(X_train, y_train, epochs=10,
                           validation_data=(X_valid, y_valid))

You can then get the best models, based on the objective metric you sepcified (here, the accuracy on the validation set).  
Run the next cell to store the 3 best models from the trials you just ran.

In [ ]:
top3_models = random_search_tuner.get_best_models(num_models=3)
best_model = top3_models[0]

Run the next 3 cells to display the hyperparameters of the 3 best models.

In [ ]:
top3_params = random_search_tuner.get_best_hyperparameters(num_trials=3)
top3_params[0].values  # best hyperparameter values

In [ ]:
top3_params[1].values 

In [ ]:
top3_params[2].values 

The tuner is guided by a so-called *oracle*, which tells it what the next trial should be. The `RandomSearchOracle` used for the `RandomSearch` tuner is just picking the next trial randomly (i.e., generating random hyperparameters values in the specified ranges). The oracle keeps track of the trials and you can ask it to give you the best trial and display a summary of this trial.

In [ ]:
best_trial = random_search_tuner.oracle.get_best_trials(num_trials=1)[0]
best_trial.summary()

**Todo**: Identify the different elements present in this summary. 

You can also access the calculated metrics individually.

In [ ]:
best_trial.metrics.get_last_value("val_accuracy")

If you are satisfied with the performance of the current best model on the validation set, you can train it on the full training set (including the validation set) evaluate it on the test set (else, you can run a few more trials to try to find a better model).

In [ ]:
best_model.fit(X_train_full, y_train_full, epochs=10)

In [ ]:
test_loss, test_accuracy = best_model.evaluate(X_test, y_test)

#### Hyperband tuner
The random tuner is quick and simple as it only explores the search space purely randomly. This is sometimes enough to get a satisfactory result, but there exists "smarter" types of search. The hyperband search, adapted from the hyperband algorithm, is roughly performing the following steps:
1) Train a lot of different models (with randomly chosen hyperparameter values within the specified ranges) for a few epochs.
2) Eliminate the worst models and keep only the $1/factor$ best models.
3) Train the remaining models for a few more epochs.
4) Repeat 2. until only 1 model is left.

The actual algorithm is slighly more sophisticated, but you do not need to know more about it for this course. If you want to learn more about, you can read the [original hyperband algorithm paper](https://jmlr.org/papers/v18/16-558.html). 

**Note**: The random and hyperband tuners are sampling a finite number of model versions to evaluate. Another option when you have a small enough search space is to explore every possible model versions within a well defined set. This is called a *grid search*. Grid search is an exhaustive search, exploring every possible combination of hyperparameters. To peform a grid search, you will need to specify all possible values you want to try for each hyperparameter (e.g., 0.1, 0.01, 0.001 for the learning rate, "sgd" and "adam" for the optimiser, etc.). You can use the [`GridSearch` tuner](https://keras.io/api/keras_tuner/tuners/grid/) with Keras to peform a grid search.

We will use the `kt.Hyperband` tuner to perform an hyperband search on the model hyperparameters, but also on the data preprocessing hyperparameters. We will have a look later at some `model.fit()` arguments such as the batch size. They can be also included in the tuning process.  
Here, we want to choose if we use normalisation or not on the input data. To include this as an hyperparameter to be tuned, we cannot directly use the `build_model()` function we defined earlier. We need to create a subclass of the `kt.HyperModel`, with `build()` and `fit()` methods. The `build()` method does the same thing as the `build_model()` function. The `fit()` method fits a compiled model and returns the `History()` object. It specifies what hyperparameters linked to the data preprocessing or the model fitting (e.g., batch size) need to be tweaked and within what range.   
For example, the following class build the same model than before, with the same hyperparameters, but it also uses a Boolean "`normalise`" hyperparameter to control whether or not to apply normalisation to the input data before fitting the model.

In [ ]:
class MyClassificationHyperModel(kt.HyperModel):
    def build(self, hp):
        return build_model(hp)

    def fit(self, hp, model, X, y, **kwargs):
        if hp.Boolean("normalize"):
            norm_layer = tf.keras.layers.Normalization()
            X = norm_layer(X)
        return model.fit(X, y, **kwargs)

The `Hyperband` tuner is defined in the following cell, in a similar fashion than the `RandomSearch` tuner. Note that some of the arguments have a different role or are specific to the `Hyperband` tuner:
- The `max_epochs` argument specifies the maximum number of epochs the best model will be trained for.   
- The `hyperband_iterations` argument specifies how many times the whole process will be repeated. Here, we set it to 2, but the docmumentation recommends to set this to "as high a value as is within your resource budget". One iteration will run approximately `max_epochs * (log(max_epochs, factor) ** 2)` epochs accross all models (about 44 epochs for the setting below).
- The `factor` argument specifies the ratio of models kept after each hyperband selection step, as described previously.

In [ ]:
hyperband_tuner = kt.Hyperband(
    MyClassificationHyperModel(), objective="val_accuracy", 
    max_epochs=10, factor=3, hyperband_iterations=2,
    overwrite=True, directory="my_fashion_mnist", project_name="hyperband")

Note: The following cell will take 15 to 20 min to run (hyperband search takes longer as it runs some intermediate trials).

In [ ]:
hyperband_tuner.search(X_train, y_train, epochs=10,
                       validation_data=(X_valid, y_valid))

You can now visualise the hyperparameters for the best model. 

In [ ]:
best_trial = hyperband_tuner.oracle.get_best_trials(num_trials=1)[0]
best_trial.summary()

**Todo**: Let's say you are happy with the performance of this model on the validation set. Train the selected model on the full train set and evaluate your trained model on the test set.

**Important note**: In a real world situation, you would only evaluate **one** final model on the test set, even if you are trying out different hyperparameter tuning methods. Else, you would get data leakage as you would be using the test set to select a model out of several possible ones produced by several hyperparameter tuning methods. 

#### Bayesian Optimisation tuner

The `Hyperband` tuner is smarter than the `RandomSearch` tuner in the way it selects what model versions to evaluate, but it still starts from randomly chosen configurations (i.e., it still explores the search space randomly). 

Some other tuning algorithms use information from past trials to *orientate* the search in a promising direction. This is the case of the hyperparameter bayesian optimisation approach you can use with the `kt.BayesianOptimization` tuner. This tuner gradually learns a probabilistic model using a *Gaussian process* to focus the search towards areas of the search space likely to hold the best hyperparameters. However, this algorithm has its own hyperparameters, `alpha` and `beta`, that need their own tuning:
- `alpha` represents the expected level of noise in the observed performances (i.e., how confident you are in the model's predictions). A high value means that you are not too confident in the model's prediction, i.e., there is likely to be variations in the observed performances (e.g., with a small dataset).
- `beta` is a balance factor between exploration and exploitation. A high value would encourage the tuner to explore more regions of the search space, while a low value would encourage it to focus more on a promising region.

You can use the `BayesianOptimization` tuner in a similar fashion as the two previous tuners.  
The following cell will take a few minutes to run.

In [ ]:
bayesian_opt_tuner = kt.BayesianOptimization(
    MyClassificationHyperModel(), objective="val_accuracy",
    max_trials=10, alpha=1e-4, beta=2.6,
    overwrite=True, directory="my_fashion_mnist", project_name="bayesian_opt")
bayesian_opt_tuner.search(X_train, y_train, epochs=10,
                          validation_data=(X_valid, y_valid))

Hyperparameter tuning is still a hard problem and an active area of research in ML and AI. There are several possible other approaches to it, including for example genetic algorithms, inspired from the theory of Evolution and natural selection. 

### I. 3. Hyperparameter tuning with PyTorch (links) 

We will not cover hyperparameter tuning using PyTorch in this tutorial, but feel free to have a look at the following tutorials if you wish to see how it can be done:
- [How to Grid Search Hyperparameters for PyTorch Models](https://machinelearningmastery.com/how-to-grid-search-hyperparameters-for-pytorch-models/)
- [Hyperparameter tuning with Ray Tune](https://pytorch.org/tutorials/beginner/hyperparameter_tuning_tutorial.html)

## II. Design choices

Now that you know how to define a search space and run different types of hyperparameters searches, let's have a look at choices you may want to include in your hyperparameter tuning.  
We will particularly cover weight initialisation, batch normalisation, gradient clipping, some regularisation techniques, different optimisers and some learning rate scheduling approaches.  
We will not spend more time on the different activation functions, but feel free to have a look a the [list of available function for Keras/TensorFlow](https://keras.io/api/layers/activations/), and experiment with them in your own time. 

### II.1. Weight initialisation

The way you initialise the weights of your network will have a great influence on the training of the model. Let's build a small neural network and try several initialisation approaches. 

First, let's reload the Fashion MNIST dataset and scale the inputs to mean 0 and standard deviation 1.

In [ ]:
fashion_mnist = tf.keras.datasets.fashion_mnist.load_data()
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist
X_train, y_train = X_train_full[:-5000], y_train_full[:-5000]
X_valid, y_valid = X_train_full[-5000:], y_train_full[-5000:]
X_train, X_valid, X_test = X_train / 255, X_valid / 255, X_test / 255

class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

pixel_means = X_train.mean(axis=0, keepdims=True)
pixel_stds = X_train.std(axis=0, keepdims=True)
X_train_scaled = (X_train - pixel_means) / pixel_stds
X_valid_scaled = (X_valid - pixel_means) / pixel_stds
X_test_scaled = (X_test - pixel_means) / pixel_stds

The following cell defines a neural networks composed of:
- 1 `Flatten` layer to convert the inputs to a vector shape.
- 2 `Dense` layers with 50 units and the weight inintialisation argument `kernel_initializer` set to `"zeros"`. This means that the weights will be initialised to 0. Note that we did not specify any activation function. In that case, no activation is applied (ie. "linear" activation: a(x) = x).
- 2 `ReLU` layers to specify the activation functions to be applied after each `Dense` layer. This is another way to specify the activation function in Keras/TensorFlow, as a seperate layer. This is closer to the syntax used in PyTorch.
- 1 `Softmax` output layer with 10 units (one for each class).

In [ ]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28]),
    tf.keras.layers.Dense(50, kernel_initializer="ones"),  # no activation
    tf.keras.layers.ReLU(), # activation as a separate layer
    tf.keras.layers.Dense(50, kernel_initializer="ones"),  # no activation
    tf.keras.layers.ReLU(), # activation as a separate layer
    tf.keras.layers.Dense(10, activation="softmax")
])

Let's compile and train this model on the Fashion MNIST dataset.

In [ ]:
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
              metrics=["accuracy"])

In [ ]:
history = model.fit(X_train_scaled, y_train, epochs=5,
                    validation_data=(X_valid_scaled, y_valid))

**Todo**: Plot the training and validation loss and accuracy and discuss what you see. Then, display the learned weights for one of the hidden layer. Comment on what you observe.

In [ ]:
pd.DataFrame(history.history).plot(
    figsize=(8, 5), xlim=[0, 4], ylim=[0, 1000], grid=True, xlabel="Epoch",
    style=["r--", "r--.", "b-", "b-*"])
plt.legend(loc="lower left")
plt.show()

In [ ]:
model.layers

In [ ]:
weights, biases = model.layers[3].get_weights() # returns weights and biases matrices for the first hidden layer as np.array 
weights

When you display the weights for one hidden layer, you will see that they are the same for each neuron in the layer (i.e., different weights have different values, but these values are the same for each neuron). All neurons in one layer are learning the same thing, which leads to the network not learning much (i.e., it is stuck on a high loss). This is because we did not *break the symmetry* of the network.  
To do so, the weights have to be initialised to small random values. Random to break the symmetry, small to avoid blowing activations when using unbounded activation functions (e.g., ReLU). There are 2 main types of initialisation:
- Xavier (also called Glorot) initialisation for Linear, tanh, sigmoid and softmax activation functions.
- He initialisation for ReLU and its variants.
These approaches can be used to generate the weight values from a uniform or a normal distribution, parameterised such that the variance of the outputs of a layer is equal to the variance of its inputs.

You can learn more about the importance of weight initialisation with by reading [this AI note from Deeplearning.ai](https://www.deeplearning.ai/ai-notes/initialization/index.html).
 
Let's try the Xavier/Glorot initialisation in the hidden layers of the previous network. You need to set `kernel_initializer="glorot_normal` (you can also use `glorot_uniform` if you want a uniform distribution instead of normal). You can read more about the different initialisation methods available in Keras/TensorFlow by reading [the documentation](https://keras.io/api/layers/initializers/).

In [ ]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28]),
    tf.keras.layers.Dense(50, kernel_initializer="glorot_normal"),  # no activation
    tf.keras.layers.ReLU(), # activation as a separate layer
    tf.keras.layers.Dense(50, kernel_initializer="glorot_normal"),  # no activation
    tf.keras.layers.ReLU(), # activation as a separate layer
    tf.keras.layers.Dense(10, activation="softmax")
])

In [ ]:
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
              metrics=["accuracy"])
history = model.fit(X_train_scaled, y_train, epochs=5,
                    validation_data=(X_valid_scaled, y_valid))

See the difference? Feel free to experiment with different initialisation methods and see what influence it has on the training. This might be another hyperparameter to try and tune!  
Using an adapted initialisation method will significantly inprove your model training and reduce the risk of exploding/vanishing gradients at the start of training.

### II.2. Batch Normalisation

Using an adapted weight initialisation approach and normalising inputs to the network usually help a lot to reduce the risk of exploring/vanishing gradients at the start of training. Howerver, it might still happens later during training as the values of the weights get updated over and over. To reduce that risk, it is possible to use *batch normalisation* with mini-batch GD. The idea is to normalise each batch of data as it goes through the network, and not only at the input. Batch normalisation can be applied before the input of each layer, or before applying the activation function in each layer. The mean and standard deviation of each mini-batch is estimated to zero-center and normalise the inputs to a layer.

There are a lot of advantages to using batch normalisation, such as:
- It speeds up the training and can improve your model performances.
- It reduces the risk of exploding/vanishing gradients during training.
- It may be used before the input layer to replace input data normalisation.
- It has a regularisation effect and reduce the need for other regularisation techniques, such as dropout.

The main drawback is that it adds a runtime overhead to the training and the predictions. The neural network makes slower predictions because of the extra computations needed to normalise the data at each layer. During training, it can be balanced by the fact that the model will likely need less epochs to reach similar performances than a design without batch normalisation. At prediction time, it is possible to alleviate this by integrating the parameters learned by the batch normalisation layer into the previous layer. 

We will not explain how batch normalisation works more in details here, but if you are interested to learn more about it, you can read the dedicated section in Chapter 11 of the "*Hands on ML*" book, or this [blog post](https://machinelearningmastery.com/batch-normalization-for-training-of-deep-neural-networks/). 

Let's modify the previous network to add `BatchNormalization` layers as the first layer of the network, and after the activation functions of the two hidden layers.

In [ ]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28]),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(50, kernel_initializer="glorot_normal"),  # no activation
    tf.keras.layers.ReLU(), # activation as a separate layer
    tf.keras.layers.BatchNormalization(), # BN after activation function
    tf.keras.layers.Dense(50, kernel_initializer="glorot_normal"),  # no activation
    tf.keras.layers.ReLU(), # activation as a separate layer
    tf.keras.layers.BatchNormalization(), # BN after activation function
    tf.keras.layers.Dense(10, activation="softmax")
])

In [ ]:
model.summary()

Let's train the model again. For a small neural network like this one, it will not change much, but it will have significant impatc if you train larger networks!

In [ ]:
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
              metrics=["accuracy"])
history_xavier = model.fit(X_train_scaled, y_train, epochs=5,
                    validation_data=(X_valid_scaled, y_valid))

You can also decide to place the `BatchNormalization` layers before the activation functions in each hidden layer, as shown below. There is debate around which approach works best, but it seems to depend on the target task. Another option to potentially try in the hyperparameter tuning phase!

In [ ]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28]),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(50, kernel_initializer="glorot_normal"),  # no activation
    tf.keras.layers.BatchNormalization(), # BN before activation function
    tf.keras.layers.ReLU(), # activation as a separate layer
    tf.keras.layers.Dense(50, kernel_initializer="glorot_normal"),  # no activation
    tf.keras.layers.BatchNormalization(), # BN before activation function
    tf.keras.layers.ReLU(), # activation as a separate layer
    tf.keras.layers.Dense(10, activation="softmax")
])

If you look at the [`BatchNormalization` layer documentation](https://keras.io/api/layers/normalization_layers/batch_normalization/), there is a number of hyperparameters you can modify. The default values are usually fine in most cases. Howerver, the `axis` hyperparameter might need to be modified if the shape of the batch is not a vector (e.g., if you apply it before the `Flatten` layer). 

### II.3. Gradient clipping

Gradient clipping is a technique used to mitigate exploding gradients. The core principle is to *clip* the gradients during training so that they never exceed some threshold. This is particularly used in *Recurrent Neural Networks* (RNNs) where using batch normalisation is tricky (ad you will see in the next tutorial).  
To use gradient clipping, you need to set the `clipvalue` or `clipnorm` argument of the optimiser when it is created.

In [ ]:
optimizer = tf.keras.optimizers.SGD(clipvalue=1.0) # option with clipvalue
# optimizer = tf.keras.optimizers.SGD(clipnorm=1.0) # option with clipnorm
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer)

With `clipvalue` set to one, the gradient will be clipped to 1 or -1 if it goes larger or smaller than these thresholds. In works well in practice, but it might change the direction of the gradient in some cases. For example, if the initial gradient vector is [0.9, 100.0], it will become [0.9, 1.0] after clipping, which directions are pretty different. If you wish to avoid changing the gradient direction, you can use the `clipnorm` argument instead. This will clip the whole gradient if its $l_2$ norm is greater then the threshold you specified. For example, with `clipnorm = 1`, the gradient vector [0.9, 100.0] will become [0.00899964, 0.9999595]. This time, the first component is almost eliminated.  
The choice between one option or the other usually comes down to what workds best in practice on your task (again!). You could integrate both options with different thresholds in your hyperparameter tuning and see what leads to better performances in the validation set.  

### II.4. Learning rate tuning and scheduling

The learning rate is in practice the most important hyperparameter to tune. It controls how large are the steps you take during the gradient descent optimisation. If it is set too high, you might risk to overshoot the optimum and diverge. If it is set too low, you will converge very slowly to the optimum.  


#### II.4.i. Tip for tuning the learning rate
One quick way to find a good value for your problem is to train your model for a few hundred iterations, starting with a very low learning rate value (e.g., $10^{-5}$) and increasing it by a constant factor each iteration up to a very large value (e.g., $10$). You can do this by multiplying the learning rate by the chosen contact factor (e.g., by $(10/10^{-5})^{1/500}$ to go from  $10^{-5}$ to $10$ in 500 iterations). If you plot the loss as a function of the learning rate (using a log scale for the learning rate), you should see it dropping at first, and go back up after a while. The optimal learning rate will be somewhere a bit lower than the point at which it starts to go back up (about 10 times lower). 

#### II.4.ii. Learning rate scheduling
Usually, it is best to use a varying learning rate, rather than a constant one. The idea is to start with a larger learning rate at the start of the training, to make fast progress towards an optimum (e.g., large loss decrease), and then reduce it as the optimisation process gets closer to converging (i.e., when progress slow down), to avoid overshooting the optimum. It can also be used to try avoiding local optima, by alternating lower and higher learning rates.  
Here are some commonly used learning rate schedules:
- **Power scheduling**: Set the learning rate $\alpha$ as a function of the iteration number $t$, following the equation $\alpha(t) = \alpha_0 / (1 + \frac{t}{s})^c$ . $\alpha_0$ is the original learning rate, the power $c$ and the steps $s$ are hyperparameters of the schedule. With $c = 1$ , the learning rate is down to $\alpha_0 / 2$ after $s$ steps (i.e., iterations), down to $\alpha_0 / 3$ after $s$ more steps, down to $\alpha_0 / 4$ after $s$ more steps, and so on. The learning rate drops quickly at first and then more and more slowly. Note that you need to tune $\alpha_0$, $s$ and possibly $c$ (usually set to 1).
- **Exponential scheduling**: Set the learning rate to $\alpha(t) = \alpha_0 \times 0.1^{\frac{t}{s}}$. The learning rate gradually drops by a factor 10 every $s$ steps. You need to tune $s$.
- **Piecewise constant scheduling**: Use a constant learning rate value for a number of epochs, then a lower value for another number of epochs, and so on. It requires tuning the right sequence of learning rates and how long each of them are used for.
- **Performance scheduling**: Measure the validation error every $N$ steps and reduce the learning rate by a factor of $\delta$ when the error decreases less quickly.
- **1cycle scheduling**: This schedule is different from the previous ones as it first increases the learning rate linearly from an initial value $\alpha_0$ to a maximum rate $\alpha_1$ in the first part of the learning, and then dropping it back to $\alpha_0$ in the second part of the learning. You can read more about this method in the [original paper it was published in](https://arxiv.org/abs/1708.07120).

There are a few other possible approaches, but the previous ones work generally well in practice. 

Let's define a power schedule using the `InverseTimeDecay` class, with `initial_learning_rate=0.01` (i.e., $\alpha_0 = 0.01$), `decay_steps=10_000` (i.e., $s = 10000$) and `decay_rate=1.0` (i.e., $C = 1$). Then, you need to pass it as an argument of the optimiser.

In [ ]:
lr_schedule = tf.keras.optimizers.schedules.InverseTimeDecay(
    initial_learning_rate=0.01,
    decay_steps=10000,
    decay_rate=1.0,
    staircase=False
)
optimizer = tf.keras.optimizers.SGD(learning_rate=lr_schedule)

Notice the `staircase` argument set to False. It means that the learning rate will decrease gradually. If it was set to True, the learning rate would decay only at discrete intervales.

Let's use a slightly larger neural network than before to try out this learning rate schedule.

In [ ]:
def build_model():
    return tf.keras.Sequential([
        tf.keras.layers.Flatten(input_shape=[28, 28]),
        tf.keras.layers.Dense(100, activation="relu",
                              kernel_initializer="he_normal"),
        tf.keras.layers.Dense(100, activation="relu",
                              kernel_initializer="he_normal"),
        tf.keras.layers.Dense(100, activation="relu",
                              kernel_initializer="he_normal"),
        tf.keras.layers.Dense(10, activation="softmax")
    ])

Let's compile and run the training with the learning rate schedule we defined. We are using a `LearningRateScheduler` callback to record the learning used during the training. 

In [ ]:
model = build_model() 
lr_scheduler = tf.keras.callbacks.LearningRateScheduler(lr_schedule)
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer,
                metrics=["accuracy"])

In [ ]:
history_power_scheduling = model.fit(X_train, y_train, epochs=10,
                                     validation_data=(X_valid, y_valid),
                                     callbacks=[lr_scheduler])

Let's now try an exponential schedule.   
**Todo**: Identify the different hyperparameters of the exponential schedule based on the equation we described before and what you now know about using a schedule with Keras/Tensorflow. 

In [ ]:
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=0.01,
    decay_steps=20000,
    decay_rate=0.1,
    staircase=False
)
optimizer = tf.keras.optimizers.SGD(learning_rate=lr_schedule)

In [ ]:
model = build_model() 
lr_scheduler = tf.keras.callbacks.LearningRateScheduler(lr_schedule)
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer,
                metrics=["accuracy"])

In [ ]:
history_exponential_scheduling = model.fit(X_train, y_train, epochs=10,
                                           validation_data=(X_valid, y_valid),
                                           callbacks=[lr_scheduler])

Let's now try a piecewise constant schedule.

In [ ]:
lr_schedule = tf.keras.optimizers.schedules.PiecewiseConstantDecay(
    boundaries=[5157, 10314],
    values=[0.01, 0.005, 0.001]
)
optimizer = tf.keras.optimizers.SGD(learning_rate=lr_schedule)

In [ ]:
model = build_model() 
lr_scheduler = tf.keras.callbacks.LearningRateScheduler(lr_schedule)
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer,
                metrics=["accuracy"])

In [ ]:
history_piecewise_scheduling = model.fit(X_train, y_train, epochs=10,
                                           validation_data=(X_valid, y_valid),
                                           callbacks=[lr_scheduler])

Run the following cell to visualise the evolution of the learning rates for each schedule. Note that we do not use lines to plot the piecewise constant schedule as it would give the false impression that there is a linear decay between some epochs. 

In [ ]:
ax = pd.DataFrame(history_power_scheduling.history["lr"]).plot(
    figsize=(8, 5), xlim=[0, 9], ylim=[0, 0.01], grid=True, ylabel = "lr", xlabel="Epoch",
    style=["r--"])
pd.DataFrame(history_exponential_scheduling.history["lr"]).plot(
    figsize=(8, 5), xlim=[0, 9], ylim=[0, 0.01], grid=True, ylabel = "lr", xlabel="Epoch",
    style=["b-"], ax=ax)
pd.DataFrame(history_piecewise_scheduling.history["lr"]).plot(
    figsize=(8, 5), xlim=[0, 9], ylim=[0, 0.01], grid=True, ylabel = "lr", xlabel="Epoch",
    style=["g*"], ax=ax)
ax.legend(["Exponential schedule", "Power schedule", "Piecewise Constant schedule"])
plt.show()

If you want to see an example of performance scheduling, you can have a look at the [Chapter 11 Notebook](https://github.com/ageron/handson-ml3/blob/main/11_training_deep_neural_networks.ipynb) of the "*Hands on ML*" book.

### II.5. Optimisers

Training large neural networks can be very slow. We have already seen multiple choices such as the way which have influence on the speed of training:
- the choice of weight initialisation,
- the choice of activation functions,
- the choice to use batch normalisation or not.

Another way to speeding up the training is to use a faster optimiser than the classic gradient descent optimiser. In this section, you will train a nerwork with the optimisers we talked about in the lecture (momentum, AdaGrad, RMSprop and Adam) and compare the results. 

First, let's build two helper functions to easily build and train a neural network with different optimisers. We are setting up a random seed to 74 to allow for repeatability. 

In [ ]:
def build_model(seed=74):
    tf.random.set_seed(seed)
    return tf.keras.Sequential([
        tf.keras.layers.Flatten(input_shape=[28, 28]),
        tf.keras.layers.Dense(100, activation="relu",
                              kernel_initializer="he_normal"),
        tf.keras.layers.Dense(100, activation="relu",
                              kernel_initializer="he_normal"),
        tf.keras.layers.Dense(100, activation="relu",
                              kernel_initializer="he_normal"),
        tf.keras.layers.Dense(10, activation="softmax")
    ])

def build_and_train_model(optimizer):
    model = build_model()
    model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer,
                  metrics=["accuracy"])
    return model.fit(X_train, y_train, epochs=10,
                     validation_data=(X_valid, y_valid))

#### SGD

Let's train the network with SGD and a learning rate of 0.001 as a baseline. 

In [ ]:
optimizer = tf.keras.optimizers.SGD(learning_rate=0.001)
history_sgd = build_and_train_model(optimizer)

#### Momentum SGD

Let's now train the network with SGD boosted with momentum. Momentum allows to accelerate the gradient descent by taking in consideration a decaying weighted sum of past gradients when taking a gradient descent step. You see that we just need to set the `momentum` argument of the SGD optimiser to turn momentum on with Keras/TensorFlow. We set `momentum=0.9` (i.e., $\beta = 0.9$) as this values work reasonably well in practice, but you could tune it if you wish to. We still use `learning_rate=0.001` to compare with the vanilla SGD. 

In [ ]:
optimizer = tf.keras.optimizers.SGD(learning_rate=0.001, momentum=0.9)
history_momentum = build_and_train_model(optimizer)

#### AdaGrad

Let's now train the network with the AdaGrad optimiser, and a learning rate of 0.001. AdaGrad adapts the learning rate based on the sum of previous squared gradients for each individual weight. Note that this time, we are using a different optimiser class (i.e., not SGD). 

In [ ]:
optimizer = tf.keras.optimizers.Adagrad(learning_rate=0.001)
history_adagrad = build_and_train_model(optimizer)

#### RMSProp

Let's now train the network with the RMSProp optimiser, still with a learning rate of 0.001. RMSProp is an extension of AdaGrad, using a decaying sum of previous squared gradients to avoid learning slowing down because of the increasing sum of squared gradients. The decay rate $\rho$ is typically set to 0.9. 

In [ ]:
optimizer = tf.keras.optimizers.RMSprop(learning_rate=0.001, rho=0.9)
history_rmsprop = build_and_train_model(optimizer)

#### ADAM

Finally, let's use ADAM to train the network, still with a learning rate of 0.001, and `beta_1=0.9` and `beta_2=0.999` (these are typical values proposed in the [original ADAM paper](https://arxiv.org/abs/1412.6980). ADAM is a combination of AdaGrad and RMSProp.

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9,
                                     beta_2=0.999)
history_adam = build_and_train_model(optimizer)

Run the 2 following cells to print the losses and accuracies resulting from ttraining the network with the different optimisers.

In [ ]:
for loss in ("loss", "val_loss"):
    plt.figure(figsize=(12, 5))
    opt_names = "SGD Momentum AdaGrad RMSProp Adam"
    for history, opt_name in zip((history_sgd, history_momentum, 
                                  history_adagrad, history_rmsprop, 
                                  history_adam,),
                                 opt_names.split()):
        plt.plot(history.history[loss], label=f"{opt_name}", linewidth=2)

    plt.grid()
    plt.xlabel("Epochs")
    plt.ylabel({"loss": "Training loss", "val_loss": "Validation loss"}[loss])
    plt.legend(loc="upper left")
    plt.axis([0, 9, 0.1, 1.5])
    plt.show()

In [ ]:
for accuracy in ("accuracy", "val_accuracy"):
    plt.figure(figsize=(12, 5))
    opt_names = "SGD Momentum AdaGrad RMSProp Adam"
    for history, opt_name in zip((history_sgd, history_momentum, 
                                  history_adagrad, history_rmsprop, 
                                  history_adam,),
                                 opt_names.split()):
        plt.plot(history.history[accuracy], label=f"{opt_name}", linewidth=2)

    plt.grid()
    plt.xlabel("Epochs")
    plt.ylabel({"accuracy": "Training accuracy", "val_accuracy": "Validation accuracy"}[accuracy])
    plt.legend(loc="upper left")
    plt.axis([0, 9, 0.1, 1])
    plt.show()

**Todo**: Discuss what insights you can get from the graphs. Which algorithms are leading to best performances?  
Note that the validation loss seems to be lower than the training loss, but this is only because it is calculated after the weights are updated, while the training loss is computed before. Therefore, there is a "lag" between the curves. This not a problem with accuracies are both are calculated after weight update.

There exists other optimisers, including quite a few versions of ADAM. You can see them used in [Chapter 11 Notebook](https://github.com/ageron/handson-ml3/blob/main/11_training_deep_neural_networks.ipynb) of the "*Hands on ML*" book.

### II.6. Regularisation techniques

Regularisation is important to reduce the risk of your model overfitting to the traning data. The overall goal is to incentivise the network not to learn a model that is too complex.  
We have discussed $l_1$ and $l_2$ regularisation in the lectures, as well as dropout and early stopping. We also saw that batch normalisation has some regularisation effect. 

#### $l_1$ and $l_2$ regularisation

$l_1$ and $l_2$ regularisation apply penalties to the weight values during weight update, to avoid weights getting to larger. $l_2$ regularisation will penalise large weight values more and small weight values less, leading to small values of weights, but never going to 0. It is sometimes called weight decay. $l_1$ regularisation will penelise all weight values uniformly, and will lead to some weights going to 0, resulting into a sparce weight matrix (i.e., with many weights equal to 0). You can use both at the same time.

With Keras/TensorFlow, $l_2$ regularisation can be directly specified when defining a layer, just like the activation function and the initialiser. 

In [ ]:
layer = tf.keras.layers.Dense(100, activation="relu",
                              kernel_initializer="he_normal",
                              kernel_regularizer=tf.keras.regularizers.l2(0.01))

Or use `l1(0.1)` for $l_1$ regularization with a factor of 0.1, or `l1_l2(0.1, 0.01)` for both $l_1$ and $l_2$ regularisation, with factors 0.1 and 0.01 respectively.

You usually want to apply the same activation function, initialiser and regulariser to several layers. Instead of having to include all of them in each layer you add, you can define you own layer with the selected arguments using `functools.partial`.

In [ ]:
from functools import partial

RegularizedDense = partial(tf.keras.layers.Dense,
                           activation="relu",
                           kernel_initializer="he_normal",
                           kernel_regularizer=tf.keras.regularizers.l2(0.01))

model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28]),
    RegularizedDense(100),
    RegularizedDense(100),
    RegularizedDense(100),
    RegularizedDense(10, activation="softmax")
])

Let's compile and train this network for  few epochs to verify that it works.

In [ ]:
optimizer = tf.keras.optimizers.SGD(learning_rate=0.02)
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer,
              metrics=["accuracy"])
history = model.fit(X_train, y_train, epochs=5,
                    validation_data=(X_valid, y_valid))

Feel free to run it for longer and compare its performance with a network without regularisation. You will specifically see the effect of regularisation if your model without it is overfitting.  
You can also play with the [TensorFlow Playground](https://playground.tensorflow.org/) to see this visually. 

**Tip regarding ADAM and $l_2$ regularisation**: Using $l_2$ regularisation with ADAM has been shown to lead in practice to less performant models than combined with SGD. AdamW is a variant of ADAM that implements weight decay in a better what than combining ADAM and $l_2$ regularisation directly.

#### Dropout

Dropout is one of the most popular regularisation methods for deep neural networks. The idea is that at each training step, every neuron in a layer (including the inputs in the input layer) have a probability $p$ to be temporarily *dropped out* (i.e., it is ignored during this training step). This incentivises the network not to realy too much on single neurons to build the model. As a result, the learned model is more robust to variations in the inputs (as they can get dreopped too during training) and generalise better. The dropout is turned off when making predictions with the learned model. 

You can use the `Dropout` layer by introducing them after each layer you want to apply dropout to. In the following code, dropout is applied to the inputs and to each hidden layer. We use a dropout rate equal to 0.2, which means that at every training iteration (every forward pass), 20% of the neurons in the layer will be dropped out randomly. You can increase this value if you notice that your model is still overfitting.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28]),
    tf.keras.layers.Dropout(rate=0.2),
    tf.keras.layers.Dense(100, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.Dropout(rate=0.2),
    tf.keras.layers.Dense(100, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.Dropout(rate=0.2),
    tf.keras.layers.Dense(10, activation="softmax")
])

Let's compile and train this network for few epochs to verify that it works.

In [ ]:
optimizer = tf.keras.optimizers.SGD(learning_rate=0.01)
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer,
              metrics=["accuracy"])
history = model.fit(X_train, y_train, epochs=5,
                    validation_data=(X_valid, y_valid))

**Tips regarding dropout**: 
- You do not have to apply dropout in every layers of your network. For example, if you want to build a model more robust to variations in the inputs, you can apply dropout in the top 1 to 3 layers of the network (excluding the output layer).
- Be careful when comparing the traning loss and validation loss if you are using dropout, as dropout is only active during training. Make sure to evaluate the training loss without dropout (i.e., after training). 

#### Early stopping

Early stopping is simple. You monitor a metric during training and stop training if this metric stopped improving (by a specified threshold). It allows to stop the training before starting to overfit. 

To implement early stopping with Keras/TensorFlow, you can use the `EarlyStopping` callback.  
`monitor='val_loss'` mean that we monitor the validation loss.  
`patience=3` means that the training will stop when there is no improvement in the loss for three consecutive epochs.

Note that we use a high learning rate to make the training stagnate or diverge quickly, to illustrate the use of early stopping. 

In [ ]:
model = build_model() 
early_stop_callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)
optimizer = tf.keras.optimizers.SGD(learning_rate=0.5)
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer,
              metrics=["accuracy"])
history = model.fit(X_train, y_train, epochs=20,
                    validation_data=(X_valid, y_valid), 
                    callbacks=[early_stop_callback])

**Todo**: Did the training stop earlier than expected? If so, check the last 3 values of the validation loss.   
Else, try to run the previous cell again. Training is a stochastic process, therefore, even with a large learning rate, some run might still converge while some will diverge. 

In practice, you would run your training with an adequate learning rate for a lot of epochs, and the validation loss will end up stagnating and start increasing when you start overfitting. Usually, you would use a version of your model saved a few epochs (e.g., 10) before the early stopping point. This is because your model has likely already started to overfit when it reaches the early stopping criterion. 

### II.7. Equivalencies in PyTorch (links)

You can find links below to PyTorch equivalent classes and functions for:
- [Weight initialisation](https://pytorch.org/docs/stable/nn.init.html)
- [Batch normalisation](https://pytorch.org/docs/stable/generated/torch.nn.functional.batch_norm.html)
- [Gradient clipping (norm version)](https://pytorch.org/docs/stable/generated/torch.nn.utils.clip_grad_norm_.html)
- [Learning rate scheduling and optimisers](https://pytorch.org/docs/stable/optim.html)
- Regularistion:
    -  $l_2$ regularisation can be used by setting the `weight_decay > 0` for any [optimiser](https://pytorch.org/docs/stable/optim.html)
    -  [Dropout layers](https://pytorch.org/docs/stable/nn.html#dropout-layers)

**Todo**: You can continue to play with the different parts of this notebook, and/or design your own neural network and try to tune a selection of hyperparameters we covered in this tutorial.